# Sudachi にユーザー辞書をコンパイルして追加する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/03_japanese_tokenization/sudachi_userdic.ipynb)

NLP クックブック 03-18 ／ 日本語の形態素解析と文分割

システム辞書で細かく分割されてしまう専門用語や、カタカナ表記から英語表記へ揃えたい表記ゆれを、ユーザー辞書（18 列の CSV）に登録して `sudachipy ubuild` でコンパイルし、分割と正規化形を単体テストで確かめたい。

In [1]:
!pip install -q sudachipy sudachidict_core

In [2]:
from sudachipy import Dictionary, SplitMode

sys_tokenizer = Dictionary().create()
test_sentences = [
    "自然言語処理では検索拡張生成やラベルバイアス問題が話題になる。",
    "形態素解析にスダチパイやフガシを使う。",
    "翻訳の品質推定にコメットキューイーを用いる。",
]

def show_tokens(tok, sentences):
    for s in sentences:
        items = [f"{m.surface()}({m.part_of_speech()[0]}-{m.part_of_speech()[1]}|{m.normalized_form()})"
                 for m in tok.tokenize(s, SplitMode.C)]
        print("入力:", s)
        print("  →", " / ".join(items))

print("=== システム辞書のみ（登録前） ===")
show_tokens(sys_tokenizer, test_sentences)

=== システム辞書のみ（登録前） ===
入力: 自然言語処理では検索拡張生成やラベルバイアス問題が話題になる。
  → 自然言語処理(名詞-普通名詞|自然言語処理) / で(助詞-格助詞|で) / は(助詞-係助詞|は) / 検索(名詞-普通名詞|検索) / 拡張(名詞-普通名詞|拡張) / 生成(名詞-普通名詞|生成) / や(助詞-副助詞|や) / ラベル(名詞-普通名詞|ラベル) / バイアス(名詞-普通名詞|バイアス) / 問題(名詞-普通名詞|問題) / が(助詞-格助詞|が) / 話題(名詞-普通名詞|話題) / に(助詞-格助詞|に) / なる(動詞-非自立可能|成る) / 。(補助記号-句点|。)
入力: 形態素解析にスダチパイやフガシを使う。
  → 形態素(名詞-普通名詞|形態素) / 解析(名詞-普通名詞|解析) / に(助詞-格助詞|に) / スダチパイ(名詞-普通名詞|スダチパイ) / や(助詞-副助詞|や) / フガシ(名詞-普通名詞|フガシ) / を(助詞-格助詞|を) / 使う(動詞-一般|使う) / 。(補助記号-句点|。)
入力: 翻訳の品質推定にコメットキューイーを用いる。
  → 翻訳(名詞-普通名詞|翻訳) / の(助詞-格助詞|の) / 品質(名詞-普通名詞|品質) / 推定(名詞-普通名詞|推定) / に(助詞-格助詞|に) / コメットキューイー(名詞-普通名詞|コメットキューイー) / を(助詞-格助詞|を) / 用いる(動詞-一般|用いる) / 。(補助記号-句点|。)


In [3]:
import json, shutil, subprocess, sys, tempfile
from pathlib import Path
import sudachidict_core, sudachipy

# （表層形、品詞大分類、品詞中分類、読み、正規化形）
user_entries = [
    ("ラベルバイアス問題", "名詞", "普通名詞", "ラベルバイアスモンダイ", "ラベルバイアス問題"),
    ("検索拡張生成", "名詞", "普通名詞", "ケンサクカクチョウセイセイ", "検索拡張生成"),
    ("スダチパイ", "名詞", "固有名詞", "スダチパイ", "SudachiPy"),
    ("フガシ", "名詞", "固有名詞", "フガシ", "fugashi"),
    ("コメットキューイー", "名詞", "固有名詞", "コメットキューイー", "COMET-QE"),
]

# Sudachi ユーザー辞書の 18 列 CSV 形式を作成して sudachipy ubuild で user.dic にコンパイルする
with tempfile.TemporaryDirectory() as tmp:
    work = Path(tmp)
    csv_path, dic_path, cfg_path = work / "user.csv", work / "user.dic", work / "sudachi.json"
    rows = [
        # 左連接ID, 右連接ID は名詞の標準的な値 (4785)、コストは既存語より優先されるよう小さめの値 (4000) にする
        ",".join([surf, "4785", "4785", "4000", surf, pos1, pos2, "一般", "*", "*", "*", rd, norm, "*", "A", "*", "*", "*"])
        for surf, pos1, pos2, rd, norm in user_entries
    ]
    csv_path.write_text("\n".join(rows) + "\n", encoding="utf-8")

    sys_dic = Path(sudachidict_core.__file__).parent / "resources" / "system.dic"
    cand_bin = Path(sys.executable).parent / "sudachipy"
    sudachipy_bin = str(cand_bin) if cand_bin.exists() else (shutil.which("sudachipy") or "sudachipy")
    subprocess.run([sudachipy_bin, "ubuild", "-s", str(sys_dic), "-o", str(dic_path), str(csv_path)], check=True)

    # 既定の sudachi.json を基に systemDict と userDict を指定する
    base_cfg = json.loads((Path(sudachipy.__file__).parent / "resources" / "sudachi.json").read_text(encoding="utf-8"))
    base_cfg["systemDict"] = str(sys_dic)
    base_cfg["userDict"] = [str(dic_path)]
    if "characterDefinitionFile" in base_cfg and not Path(base_cfg["characterDefinitionFile"]).is_absolute():
        base_cfg["characterDefinitionFile"] = str(Path(sudachipy.__file__).parent / "resources" / base_cfg["characterDefinitionFile"])
    cfg_path.write_text(json.dumps(base_cfg, ensure_ascii=False, indent=2), encoding="utf-8")

    user_tokenizer = Dictionary(config_path=str(cfg_path)).create()
    print("=== ユーザー辞書適用後 ===")
    show_tokens(user_tokenizer, test_sentences)

    # 単体テスト（回帰テスト）: 期待どおりの分割と正規化形になっているか確認する
    cases = [
        ("ラベルバイアス問題を学ぶ", "ラベルバイアス問題", "ラベルバイアス問題"),
        ("検索拡張生成の精度を上げる", "検索拡張生成", "検索拡張生成"),
        ("形態素解析にスダチパイを使う", "スダチパイ", "SudachiPy"),
        ("ラッパーのフガシを試す", "フガシ", "fugashi"),
        ("コメットキューイーで評価する", "コメットキューイー", "COMET-QE"),
    ]
    for sent, expected_surface, expected_norm in cases:
        matched = [m for m in user_tokenizer.tokenize(sent, SplitMode.C) if m.surface() == expected_surface]
        assert matched, f"分割失敗: {sent!r} に {expected_surface!r} が見つかりません"
        assert matched[0].normalized_form() == expected_norm, f"正規化不一致: {matched[0].normalized_form()}"
    print(f"\n単体テスト: {len(cases)}/{len(cases)} 件合格")

/tmp/tmpk3dz16qa/user.csv -> 5 in 0.00 sec
validate -> 5 in 0.00 sec
pos_table -> 2 in 0.00 sec
conn_matrix -> 6 in 0.00 sec
trie -> 1028 in 0.00 sec
word_id table -> 29 in 0.00 sec
word_params -> 34 in 0.00 sec
wordinfo_offsets -> 20 in 0.00 sec
wordinfos (copy only) -> 230 in 0.00 sec
=== ユーザー辞書適用後 ===
入力: 自然言語処理では検索拡張生成やラベルバイアス問題が話題になる。
  → 自然言語処理(名詞-普通名詞|自然言語処理) / で(助詞-格助詞|で) / は(助詞-係助詞|は) / 検索拡張生成(名詞-普通名詞|検索拡張生成) / や(助詞-副助詞|や) / ラベルバイアス問題(名詞-普通名詞|ラベルバイアス問題) / が(助詞-格助詞|が) / 話題(名詞-普通名詞|話題) / に(助詞-格助詞|に) / なる(動詞-非自立可能|成る) / 。(補助記号-句点|。)
入力: 形態素解析にスダチパイやフガシを使う。
  → 形態素(名詞-普通名詞|形態素) / 解析(名詞-普通名詞|解析) / に(助詞-格助詞|に) / スダチパイ(名詞-固有名詞|SudachiPy) / や(助詞-副助詞|や) / フガシ(名詞-固有名詞|fugashi) / を(助詞-格助詞|を) / 使う(動詞-一般|使う) / 。(補助記号-句点|。)
入力: 翻訳の品質推定にコメットキューイーを用いる。
  → 翻訳(名詞-普通名詞|翻訳) / の(助詞-格助詞|の) / 品質(名詞-普通名詞|品質) / 推定(名詞-普通名詞|推定) / に(助詞-格助詞|に) / コメットキューイー(名詞-固有名詞|COMET-QE) / を(助詞-格助詞|を) / 用いる(動詞-一般|用いる) / 。(補助記号-句点|。)

単体テスト: 5/5 件合格


- Sudachi のユーザー辞書は 18 列の CSV で表層形・連接コスト ID・生起コスト・品詞・読み・正規化形・分割タイプ（`A`/`B`/`C` のどの単位で登録するか）などを指定し、`sudachipy ubuild` でバイナリの `user.dic` に変換してから設定 JSON の `userDict` に登録する。
- 名詞として追加する場合は左・右連接 ID に標準的な名詞の ID (`4785`) を流用でき、生起コストを小さめ（`4000` など）にするとシステム辞書の細かな分割（「ラベル / バイアス / 問題」「検索 / 拡張 / 生成」）より 1 語としての分割が選ばれやすくなる。
- 表記ゆれ（「スダチパイ」→ `SudachiPy`、「フガシ」→ `fugashi`）を正規化形欄に書いておくと、`m.normalized_form()` で表記を統一できる。辞書に語を足したときは、想定外の文で誤分割が起きないかも含めて単体テストで継続的に検証する。
- 設定不要で手早く簡易 CSV を試したい場合は、この章の「Janome にユーザー辞書を追加する」も使える。